In [ ]:
"""総合演習2 社会系シミュレーション 第4回 (2026-12-02)

SIRモデルの可動エージェントシミュレーション (ヒートマップ)
"""

import random
from typing import Final, Self

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib.artist import Artist


# ========== パラメータ ==========
NUM_AGENTS: Final[int] = 100        # エージェントの数
NUM_SECTIONS_X: Final[int] = 3      # x方向の区画の数
NUM_SECTIONS_Y: Final[int] = 3      # y方向の区画の数
X_MIN: Final[float] = 0             # x座標の最小値
X_MAX: Final[float] = 10            # x座標の最大値
Y_MIN: Final[float] = 0             # y座標の最小値
Y_MAX: Final[float] = 10            # y座標の最大値
DX_ABS: Final[float] = 0.05         # x方向の移動距離の上限
DY_ABS: Final[float] = 0.05         # y方向の移動距離の上限

INFECTION_DIST: Final[float] = 0.5  # 感染可能な距離
INFECTION_PROB: Final[float] = 0.1  # 感染確率
RECOVERY_STEP: Final[int] = 1000    # 感染から回復までのステップ数

NUM_STEPS: Final[int] = 5000        # 時間ステップ数
PLOT_INTERVAL: Final[int] = 50      # 可視化するステップ間隔
# ==============================


LX: Final[float] = X_MAX - X_MIN
LY: Final[float] = Y_MAX - Y_MIN
MARGIN: Final[float] = 0.1

STATE_COLOR: Final[dict[str, str]] = {
    'susceptible': 'blue',
    'infected': 'red',
    'recovered': 'green'
}
STATE_MARKER: Final[dict[str, str]] = {
    'susceptible': 'o',
    'infected': '^',
    'recovered': 's'
}


class Section:
    """区画を表すクラス

    Attributes
    ----------
    id : int
        区画のID
    range_x : tuple[float, float]
        区画のx座標の範囲 (x_min, x_max)
    range_y : tuple[float, float]
        区画のy座標の範囲 (y_min, y_max)
    """

    def __init__(self, id: int, range_x: tuple[float, float], range_y: tuple[float, float]) -> None:
        """Sectionクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        id : int
            区画のID
        range_x : tuple[float, float]
            区画のx座標の範囲 (x_min, x_max)
        range_y : tuple[float, float]
            区画のy座標の範囲 (y_min, y_max)
        """

        self.id = id
        self.range_x = range_x
        self.range_y = range_y


class MovableAgent:
    """可動エージェントを表すクラス

    Attributes
    ----------
    x : float
        エージェントのx座標
    y : float
        エージェントのy座標
    status : str
        エージェントの感染状態 ('susceptible', 'infected', 'recovered')
    infected_step : int
        感染してからのステップ数
    section_id : int | None
        エージェントが所属する区画のID
    """

    def __init__(self, init_x: float, init_y: float, init_status: str) -> None:
        """MovableAgentクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        init_x : float
            エージェントの初期x座標
        init_y : float
            エージェントの初期y座標
        init_status : str
            エージェントの初期感染状態 ('susceptible', 'infected', 'recovered')
        section_id : int
            エージェントが所属する区画のID
        """

        self.x: float = init_x
        self.y: float = init_y
        self.status: str = init_status
        self.infected_step: int = 0
        self.section_id: int 

    def move(self) -> None:
        """エージェントを1ステップ移動させる

        Notes
        -----
        エージェントの移動はランダムに決定
        """

        dx: float = DX_ABS * 2*(random.random() - 0.5)
        dy: float = DY_ABS * 2*(random.random() - 0.5)

        self.x += dx
        self.y += dy

        # 範囲外に出たら範囲内に戻す
        if self.x < X_MIN:
            self.x = X_MIN
        elif self.x > X_MAX:
            self.x = X_MAX
        if self.y < Y_MIN:
            self.y = Y_MIN
        elif self.y > Y_MAX:
            self.y = Y_MAX

    def change_status(self, infected_agents: list[Self]) -> None:
        """エージェントの感染状態を更新する

        Parameters
        ----------
        infected_agents : list[Self]
            感染しているエージェントのリスト
        """

        if self.status == 'susceptible':
            for other in infected_agents:
                if self.calc_distance(other) <= INFECTION_DIST:
                    if random.random() < INFECTION_PROB:
                        self.status = 'infected'
                        break

        if self.status == 'infected':
            self.infected_step += 1
            if self.infected_step >= RECOVERY_STEP:
                self.status = 'recovered'

    def calc_distance(self, other: Self) -> float:
        """他のエージェントとの距離を計算する

        Parameters
        ----------
        other : MovableAgent
            距離を計算する相手のエージェント

        Returns
        -------
        distance : float
            他のエージェントとの距離
        """

        difference_x: float = self.x - other.x
        difference_y: float = self.y - other.y
        distance: float = (difference_x**2 + difference_y**2)**0.5

        return distance

    def calc_section(self, sections: list[Section]) -> None:
        """エージェントが所属する区画を計算する

        Parameters
        ----------
        sections : list[Section]
            全ての区画を格納するリスト
        """

        for section in sections:
            if section.range_x[0] <= self.x <= section.range_x[1]:
                if section.range_y[0] <= self.y <= section.range_y[1]:
                    self.section_id = section.id
                    break


class MovableAgentSystem:
    """可動エージェント全体を表すクラス

    Attributes
    ----------
    num_agents : int
        エージェントの数
    agents : list[MovableAgent]
        全ての可動エージェントを格納するリスト
    timestep : int
        現在の時間ステップ
    num_sections_x : int
        x方向の区画の数
    num_sections_y : int
        y方向の区画の数
    num_sections : int
        区画の総数
    sections : list[Section]
        全ての区画を格納するリスト
    data_infected : list[int]
        区画ごとの感染者数
    data_susceptible : list[int]
        区画ごとの感受性者数
    data_recovered : list[int]
        区画ごとの回復者数
    """

    def __init__(self, num_agents: int, num_sections_x: int, num_sections_y: int) -> None:
        """MovableAgentSystemクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        num_agents : int
            エージェントの数
        num_sections_x : int
            x方向の区画の数
        num_sections_y : int
            y方向の区画の数

        Notes
        -----
        エージェントの初期位置はランダムに設定
        """

        self.num_agents: int = num_agents
        self.agents: list[MovableAgent] = []
        self.timestep: int = 0
        self.num_sections_x: int = num_sections_x
        self.num_sections_y: int = num_sections_y
        self.num_sections: int = num_sections_x * num_sections_y
        self.sections: list[Section] = []
        self.data_infected: list[int] = []
        self.data_susceptible: list[int] = []
        self.data_recovered: list[int] = []

        for i in range(num_agents):
            init_x: float = X_MIN + (X_MAX - X_MIN) * random.random()
            init_y: float = Y_MIN + (Y_MAX - Y_MIN) * random.random()

            init_status: str = 'susceptible'
            if i == 0:
                init_status = 'infected'

            self.agents.append(MovableAgent(init_x, init_y, init_status))

        for j in range(self.num_sections_y):
            for i in range(self.num_sections_x):
                section_id: int = i + self.num_sections_x * j
                range_x: tuple[float, float] = (
                    X_MIN + i * (X_MAX - X_MIN) / self.num_sections_x,
                    X_MIN + (i + 1) * (X_MAX - X_MIN) / self.num_sections_x)
                range_y: tuple[float, float] = (
                    Y_MIN + j * (Y_MAX - Y_MIN) / self.num_sections_y,
                    Y_MIN + (j + 1) * (Y_MAX - Y_MIN) / self.num_sections_y)
                self.sections.append(
                    Section(section_id, range_x, range_y))

        for agent in self.agents:
            agent.calc_section(self.sections)

        self.data_susceptible, self.data_infected, self.data_recovered \
            = self.count_status()

    def time_evolution(self) -> None:
        """システムを1ステップ時間発展させる"""

        infected_agents: list[MovableAgent] = [
            agent for agent in self.agents if agent.status == 'infected']
        for agent in self.agents:
            agent.change_status(infected_agents)

        for agent in self.agents:
            agent.move()
            agent.calc_section(self.sections)

        self.data_susceptible, self.data_infected, self.data_recovered \
            = self.count_status()

        self.timestep += 1

    def count_status(self) -> tuple[list[int], list[int], list[int]]:
        """各区画ごとのエージェントの感染状態をカウントする"""

        data_susceptible: list[int] = [0] * self.num_sections
        data_infected: list[int] = [0] * self.num_sections
        data_recovered: list[int] = [0] * self.num_sections
        for agent in self.agents:
            if agent.status == 'susceptible':
                data_susceptible[agent.section_id] += 1
            elif agent.status == 'infected':
                data_infected[agent.section_id] += 1
            elif agent.status == 'recovered':
                data_recovered[agent.section_id] += 1

        return data_susceptible, data_infected, data_recovered

    def visualize(self) -> list[Artist]:
        """各タイムステップでのエージェントの位置を可視化する

        Returns
        -------
        frame : list[Artist]
            可視化結果のArtistオブジェクトのリスト

        Notes
        -----
        matplotlib.animation.ArtistAnimationを使用
        """

        frame: list[Artist] = [
            ax[0].text(X_MIN, Y_MAX+1.5*MARGIN*LY, f'time step = {self.timestep}')
        ]
        for agent in self.agents:
            frame += ax[0].plot(agent.x, agent.y,
                             color=STATE_COLOR[agent.status],
                             marker=STATE_MARKER[agent.status])

        data: list[list[float]] = [
            [0] * self.num_sections_x for _ in range(self.num_sections_y)]
        for j in range(self.num_sections_y):
            for i in range(self.num_sections_x):
                section_id = i + self.num_sections_x * j
                num_agents_in_section: int = (
                    self.data_infected[section_id] 
                    + self.data_susceptible[section_id] 
                    + self.data_recovered[section_id]
                )
                if num_agents_in_section == 0:
                    data[j][i] = 0
                else:
                    data[j][i] = (
                        4 *self.data_infected[section_id] 
                        * self.data_susceptible[section_id]
                        / (num_agents_in_section**2)
                    )
        im: Artist = ax[1].imshow(data, cmap='Reds', vmin=0, vmax=1,
                              origin='lower',
                              extent=[X_MIN, X_MAX, Y_MIN, Y_MAX])
        frame.append(im)

        return frame


if __name__ == '__main__':

    # 可動エージェントと区画を初期化
    system: MovableAgentSystem = MovableAgentSystem(
        NUM_AGENTS, NUM_SECTIONS_X, NUM_SECTIONS_Y)

    # 可視化の準備
    fig, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=150)
    for a in ax:
        a.set_xlabel('x')
        a.set_ylabel('y')
        a.set_xlim(X_MIN-MARGIN*LX, X_MAX+MARGIN*LX)
        a.set_ylim(Y_MIN-MARGIN*LY, Y_MAX+MARGIN*LY)
        a.grid()
        a.set_aspect('equal')
    for status in STATE_COLOR.keys():
        ax[0].plot([], [],
                color=STATE_COLOR[status], marker=STATE_MARKER[status], linestyle='None', label=status)
    ax[0].legend(loc='upper right', bbox_to_anchor=(1.1, 1.15))
    ax[1].set_title('4IS/N^2')

    # 壁の描画
    for a in ax:
        a.axvspan(X_MIN-MARGIN*LX, X_MIN, color='gray')
        a.axvspan(X_MAX, X_MAX+MARGIN*LX, color='gray')
        a.axhspan(Y_MIN-MARGIN*LY, Y_MIN, color='gray')
        a.axhspan(Y_MAX, Y_MAX+MARGIN*LY, color='gray')

    # 初期時刻の可視化
    artists: list[list[Artist]] = []
    artists.append(system.visualize())

    # 時間発展
    for _ in range(NUM_STEPS):
        system.time_evolution()

        if system.timestep % PLOT_INTERVAL == 0:
            artists.append(system.visualize())

    # アニメーションの作成
    anim = animation.ArtistAnimation(fig, artists)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))
    anim.save('./social_4th_viz2.gif', writer="pillow")